# Prep: Winnipeg Mosquito Trap Counts

**Why this dataset:** eleven complete seasons, one clear row (a count day), and questions people in Winnipeg actually
ask. **The questions it's prepared for:** which summers are the worst, when in the summer, and which side of the city.

In [1]:
import pandas as pd

traps = pd.read_csv("data/raw/traps-2015-2025.csv")
traps.shape

(880, 44)

## Step 1: dates

**What:** read `Count Date` day first, and add a `year` column. **Why:** the file writes `13/05/2015`; read month first
it stops at May 13, and on days 1 to 12 it would swap day and month without an error. **Cost:** nothing.
**Check:** every count day falls between May and October.

In [2]:
traps["date"] = pd.to_datetime(traps["Count Date"], dayfirst=True)
traps["year"] = traps["date"].dt.year
assert traps["date"].dt.month.between(5, 10).all()
traps[["Count Date", "date", "year"]].head(3)

,Count Date,date,year
0,04/05/2015,2015-05-04,2015
1,06/05/2015,2015-05-06,2015
2,08/05/2015,2015-05-08,2015


## Step 2: `no data` becomes missing

**What:** in the trap columns, `no data` becomes a real missing value, and the columns become numbers. **Why:** it
means the trap wasn't counted, not zero mosquitoes; as text, the columns can't be averaged. **Cost:** none:
nothing is filled in or dropped. **Check:** exactly 579 missing values, and no
text columns left except the date.

In [3]:
trap_cols = [c for c in traps.columns if traps[c].dtype == "object" and c != "Count Date"]
before = (traps[trap_cols] == "no data").sum().sum()
traps[trap_cols] = traps[trap_cols].replace("no data", pd.NA).apply(pd.to_numeric)
print("no data before:", before, " missing after:", traps[trap_cols].isna().sum().sum())
traps.dtypes.value_counts()

no data before: 579  missing after: 579


float64           37
int64              6
object             1
datetime64[ns]     1
int32              1
Name: count, dtype: int64